# 05 — Preparation of the Facebook 'datacenter' Query Corpus


    **Research objective.** Prepare the consistent `datacenter` keyword export for longitudinal, discourse, engagement, and legislative-linkage analyses.

    **Input.** `../data/raw/facebook_datacenter_2024_2026.csv`.

    **Methods.** Parse UTC timestamps; verify `datacenter` and `datacenters` using word-boundary regular expressions; apply a transparent policy-term dictionary; calculate engagement as reactions plus comments plus shares; normalize text; audit post-ID, text, and owner duplication; and construct both a January 2024–August 2026 longitudinal corpus and an April 1–August 17, 2026 discourse subset from the same query. Views are excluded because coverage is inconsistent. All cleaning and validation functions used in this stage are defined directly in this notebook.

    **Outputs.** Compressed longitudinal and 2026 discourse datasets, `facebook_query_coverage.csv`, `facebook_filtering_flow.csv`, duplicate diagnostics, and blank manual-review sheets.

In [1]:
from __future__ import annotations

import json

import math

import re

from pathlib import Path

import matplotlib

import matplotlib.dates as mdates

import matplotlib.pyplot as plt

import numpy as np

import pandas as pd

from scipy.stats import spearmanr

from sklearn.compose import ColumnTransformer

from sklearn.decomposition import LatentDirichletAllocation

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

from sklearn.model_selection import (
    GroupShuffleSplit,
    StratifiedKFold,
    cross_val_predict,
    train_test_split,
)

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder, StandardScaler

matplotlib.use("Agg")

SEED = 149

PHRASE_RE = re.compile(
    r"\b(?:data\s*cent(?:er|re)s?|datacent(?:er|re)s?)\b", re.IGNORECASE
)

POLICY_TERMS = [
    r"\bbills?\b", r"\blegislat(?:ion|ive|or|ors)\b", r"\blawmakers?\b",
    r"\bsenate\b", r"\bassembly\b", r"\bhouse bill\b", r"\bgovernor\b",
    r"\bcommission\b", r"\blaws?\b", r"\bregulat(?:ion|e|es|ed|ory)\b",
    r"\bzoning\b", r"\bpermits?\b", r"\bordina(?:nce|nces)\b",
    r"\bmoratorium\b", r"\bbans?\b", r"\btax (?:credit|break|exemption)s?\b",
    r"\bincentives?\b", r"\bsubsid(?:y|ies)\b", r"\butilit(?:y|ies)\b",
    r"\bratepayers?\b", r"\belectricity rates?\b", r"\bwater use\b",
    r"\benvironmental review\b", r"\bpublic hearings?\b", r"\bdisclos(?:ure|e)\b",
    r"\btransparency\b",
]

POLICY_RE = re.compile("|".join(POLICY_TERMS), re.IGNORECASE)

FRAME_PATTERNS = {
    "Energy and utility costs": [
        r"\belectric(?:ity|al)\b", r"\bpower\b", r"\bgrid\b", r"\butilit(?:y|ies)\b",
        r"\bratepayers?\b", r"\brates?\b", r"\bmegawatts?\b", r"\btransmission\b",
        r"\binterconnection\b", r"\benergy\b",
    ],
    "Water and environmental effects": [
        r"\bwater\b", r"\benvironment(?:al)?\b", r"\bemissions?\b", r"\bcarbon\b",
        r"\bpollution\b", r"\bclimate\b", r"\bsustainab(?:le|ility)\b", r"\baquifer\b",
    ],
    "Economic development and jobs": [
        r"\bjobs?\b", r"\bemployment\b", r"\beconomic development\b", r"\binvest(?:ment|s|ed)\b",
        r"\bbusiness(?:es)?\b", r"\bconstruction\b", r"\bgrowth\b", r"\brevenue\b",
    ],
    "Taxes, incentives, and subsidies": [
        r"\btax(?:es|ation)?\b", r"\btax (?:credit|break|exemption)s?\b", r"\bincentives?\b",
        r"\bsubsid(?:y|ies)\b", r"\babatement\b", r"\bpublic funds?\b",
    ],
    "Regulation and community control": [
        r"\bregulat(?:ion|e|ed|ory)\b", r"\bzoning\b", r"\bpermits?\b", r"\bordina(?:nce|nces)\b",
        r"\bmoratorium\b", r"\bbans?\b", r"\bcommunity\b", r"\blocal control\b",
        r"\bpublic hearings?\b", r"\bdisclos(?:ure|e)\b", r"\btransparency\b",
    ],
    "AI growth and technological competition": [
        r"\bartificial intelligence\b", r"\bAI\b", r"\bcloud\b", r"\bcompute\b",
        r"\bdigital infrastructure\b", r"\btechnology\b", r"\binnovation\b", r"\bhyperscale\b",
    ],
}

FRAME_REGEX = {
    name: re.compile("|".join(patterns), re.IGNORECASE)
    for name, patterns in FRAME_PATTERNS.items()
}

def ensure_dirs(root: Path) -> None:
    for relative in ["data/processed", "output/figures", "output/tables"]:
        (root / relative).mkdir(parents=True, exist_ok=True)

def save_table(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)
    print(f"Saved {len(frame):,} rows -> {path}")

def save_gzip(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False, compression="gzip")
    print(f"Saved {len(frame):,} rows -> {path}")

def normalize_text_value(value: object) -> str:
    """Normalize text for duplicate detection while preserving original text elsewhere."""
    text = "" if pd.isna(value) else str(value).lower()
    text = re.sub(r"https?://\S+", " ", text)
    text = re.sub(r"\b(?:utm_[a-z_]+|fbclid)=[^\s&]+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def phrase_flag(text: object) -> bool:
    return bool(PHRASE_RE.search("" if pd.isna(text) else str(text)))

def policy_flag(text: object) -> bool:
    return bool(POLICY_RE.search("" if pd.isna(text) else str(text)))

def add_frame_flags(frame: pd.DataFrame, text_col: str = "text") -> pd.DataFrame:
    out = frame.copy()
    text = out[text_col].fillna("").astype(str)
    for name, regex in FRAME_REGEX.items():
        out[f"frame_{slug(name)}"] = text.str.contains(regex, na=False)
    return out

def slug(value: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", value.lower()).strip("_")

def load_facebook(path: Path, source_query: str, source_file: str, source_start: str, source_end: str) -> pd.DataFrame:
    frame = pd.read_csv(path, low_memory=False)
    frame["creation_time"] = pd.to_datetime(frame["creation_time"], errors="coerce", utc=True)
    frame["source_query"] = source_query
    frame["source_file"] = source_file
    frame["source_start_date"] = source_start
    frame["source_end_date"] = source_end
    frame["normalized_text"] = frame["text"].map(normalize_text_value)
    frame["contains_data_center_term"] = frame["text"].map(phrase_flag)
    frame["policy_related"] = frame["text"].map(policy_flag)
    numeric = ["statistics.reaction_count", "statistics.comment_count", "statistics.share_count"]
    for column in numeric:
        frame[f"raw_missing_{slug(column)}"] = frame[column].isna()
        frame[column] = pd.to_numeric(frame[column], errors="coerce").fillna(0)
    frame["total_reactions"] = frame["statistics.reaction_count"]
    frame["comments"] = frame["statistics.comment_count"]
    frame["shares"] = frame["statistics.share_count"]
    frame["total_engagement"] = frame["total_reactions"] + frame["comments"] + frame["shares"]
    frame["log_engagement"] = np.log1p(frame["total_engagement"])
    return add_frame_flags(frame)

def _facebook_export_row(name: str, frame: pd.DataFrame) -> dict:
    english = frame.lang.eq("en")
    relevant = english & frame.contains_data_center_term
    policy = relevant & frame.policy_related
    return {
        "export": name, "rows": len(frame), "columns": frame.shape[1],
        "min_creation_time_utc": frame.creation_time.min(), "max_creation_time_utc": frame.creation_time.max(),
        "unique_post_ids": frame.id.nunique(), "duplicate_post_ids": frame.id.duplicated().sum(),
        "english_rows": english.sum(), "exact_term_english_rows": relevant.sum(),
        "policy_english_rows": policy.sum(), "unique_english_normalized_texts": frame.loc[english & frame.normalized_text.ne(""), "normalized_text"].nunique(),
        "unique_owners": frame["post_owner.id"].nunique(),
        "missing_reactions": frame["raw_missing_statistics_reaction_count"].sum(),
        "missing_comments": frame["raw_missing_statistics_comment_count"].sum(),
        "missing_shares": frame["raw_missing_statistics_share_count"].sum(),
        "missing_views": frame["statistics.views"].isna().sum(),
    }

def _manual_fb_sample(frame: pd.DataFrame, sample_name: str) -> pd.DataFrame:
    pieces = []
    for value, label in [(True, "policy_related"), (False, "broad_nonpolicy")]:
        pool = frame.loc[frame.policy_related.eq(value)].copy()
        pieces.append(pool.sample(n=min(50, len(pool)), random_state=SEED))
    manual = pd.concat(pieces, ignore_index=True)
    manual["review_stratum"] = np.where(manual.policy_related, "policy_related", "broad_nonpolicy")
    manual["sample_name"] = sample_name
    manual["manual_relevant"] = ""
    manual["manual_policy_related"] = ""
    manual["manual_notes"] = ""
    columns = ["sample_name","review_stratum","id","creation_time","post_owner.name","content_type","text","contains_data_center_term","policy_related","total_engagement","manual_relevant","manual_policy_related","manual_notes"]
    return manual[columns]

def run_facebook_cleaning(root: Path) -> dict:
    ensure_dirs(root)
    posts = load_facebook(
        root / "data/raw/facebook_datacenter_2024_2026.csv", "datacenter",
        "facebook_datacenter_2024_2026.csv", "2024-01-01", "2026-08-17",
    )
    print(f"Datacenter-query export: {len(posts):,} rows; {posts.creation_time.min()} to {posts.creation_time.max()}")
    coverage = pd.DataFrame([_facebook_export_row("datacenter", posts)])
    coverage["analysis_design"] = "single consistent query"
    save_table(coverage, root / "output/tables/facebook_query_coverage.csv")
    for variable, filename in [
        ("lang", "facebook_language_distribution.csv"),
        ("post_owner.type", "facebook_owner_type_distribution.csv"),
        ("content_type", "facebook_content_type_distribution.csv"),
    ]:
        distribution = posts[variable].fillna("<missing>").value_counts().rename("count").reset_index().rename(columns={variable: "value"})
        distribution["query"] = "datacenter"
        distribution["variable"] = variable
        save_table(distribution[["query","variable","value","count"]], root / f"output/tables/{filename}")

    flow = []
    def flow_row(sample, step, frame):
        flow.append({"sample": sample, "step": step, "rows": len(frame), "unique_ids": frame.id.nunique(), "unique_normalized_texts": frame.loc[frame.normalized_text.ne(""), "normalized_text"].nunique()})

    flow_row("Facebook 'datacenter' Query Corpus", "raw_query_rows", posts)
    english = posts.loc[posts.lang.eq("en")].copy()
    flow_row("Facebook 'datacenter' Query Corpus", "english", english)
    longitudinal = english.loc[english.contains_data_center_term].copy()
    longitudinal["id_duplicate_count"] = longitudinal.groupby("id")["id"].transform("size")
    longitudinal["duplicate_count"] = longitudinal.groupby("normalized_text")["normalized_text"].transform("size")
    flow_row("Facebook 'datacenter' Query Corpus", "exact_term_relevant", longitudinal)
    longitudinal_policy = longitudinal.loc[longitudinal.policy_related].copy()
    flow_row("Facebook 'datacenter' Query Corpus", "policy_related", longitudinal_policy)
    longitudinal_text = longitudinal.sort_values("creation_time").drop_duplicates("normalized_text", keep="first").copy()
    flow_row("Facebook 'datacenter' Query Corpus", "normalized_text_deduplicated", longitudinal_text)
    owner_duplicates = longitudinal.groupby(["source_query","post_owner.id"], dropna=False).agg(
        owner_name=("post_owner.name", "first"), qualifying_rows=("id", "size"),
        unique_post_ids=("id", "nunique"), unique_normalized_texts=("normalized_text", "nunique"),
    ).reset_index()
    owner_duplicates["repeated_id_rows"] = owner_duplicates.qualifying_rows-owner_duplicates.unique_post_ids
    owner_duplicates["repeated_text_rows"] = owner_duplicates.qualifying_rows-owner_duplicates.unique_normalized_texts
    save_table(owner_duplicates.sort_values("repeated_text_rows", ascending=False), root / "output/tables/facebook_duplicate_owner_counts.csv")

    discourse_start = pd.Timestamp("2026-04-01", tz="UTC")
    discourse_end = pd.Timestamp("2026-08-18", tz="UTC")
    discourse_2026 = longitudinal.loc[
        longitudinal.creation_time.ge(discourse_start) & longitudinal.creation_time.lt(discourse_end)
    ].copy()
    flow_row("Facebook 'datacenter' Query Corpus, April–August 2026", "relevant", discourse_2026)
    discourse_policy = discourse_2026.loc[discourse_2026.policy_related].copy()
    flow_row("Facebook 'datacenter' Query Corpus, April–August 2026", "policy_related", discourse_policy)
    discourse_text = discourse_2026.sort_values("creation_time").drop_duplicates("normalized_text", keep="first").copy()
    flow_row("Facebook 'datacenter' Query Corpus, April–August 2026", "normalized_text_deduplicated", discourse_text)
    discourse_policy_text = discourse_text.loc[discourse_text.policy_related].copy()
    flow_row("Facebook 'datacenter' Query Corpus, April–August 2026", "policy_and_text_deduplicated", discourse_policy_text)
    save_table(pd.DataFrame(flow), root / "output/tables/facebook_filtering_flow.csv")

    save_gzip(longitudinal, root / "data/processed/facebook_datacenter_longitudinal_2024_2026.csv.gz")
    save_gzip(longitudinal_policy, root / "data/processed/facebook_datacenter_longitudinal_2024_2026_policy.csv.gz")
    save_gzip(longitudinal_text, root / "data/processed/facebook_datacenter_longitudinal_2024_2026_text_deduplicated.csv.gz")
    save_gzip(discourse_2026, root / "data/processed/facebook_datacenter_discourse_2026.csv.gz")
    save_gzip(discourse_policy, root / "data/processed/facebook_datacenter_discourse_2026_policy.csv.gz")
    save_gzip(discourse_text, root / "data/processed/facebook_datacenter_discourse_2026_text_deduplicated.csv.gz")
    save_gzip(discourse_policy_text, root / "data/processed/facebook_datacenter_discourse_2026_policy_text_deduplicated.csv.gz")
    save_table(_manual_fb_sample(longitudinal, "Facebook 'datacenter' Query Corpus, 2024–2026"), root / "output/tables/facebook_manual_validation_longitudinal.csv")
    save_table(_manual_fb_sample(discourse_2026, "Facebook 'datacenter' Query Corpus, April–August 2026"), root / "output/tables/facebook_manual_validation_2026_discourse.csv")
    return {
        "raw_rows": len(posts), "english_rows": int(posts.lang.eq("en").sum()),
        "longitudinal": len(longitudinal), "longitudinal_policy": len(longitudinal_policy),
        "discourse_2026": len(discourse_2026), "discourse_2026_policy": len(discourse_policy),
        "discourse_2026_text_unique": len(discourse_text),
    }

ROOT = Path("..")
print("Random seed:", SEED)

Random seed: 149


In [2]:
results = run_facebook_cleaning(ROOT)
results

Datacenter-query export: 66,055 rows; 2024-01-01 00:55:28+00:00 to 2026-08-17 06:06:30+00:00
Saved 1 rows -> ../output/tables/facebook_query_coverage.csv
Saved 96 rows -> ../output/tables/facebook_language_distribution.csv
Saved 2 rows -> ../output/tables/facebook_owner_type_distribution.csv
Saved 8 rows -> ../output/tables/facebook_content_type_distribution.csv


Saved 5,831 rows -> ../output/tables/facebook_duplicate_owner_counts.csv
Saved 9 rows -> ../output/tables/facebook_filtering_flow.csv


Saved 24,915 rows -> ../data/processed/facebook_datacenter_longitudinal_2024_2026.csv.gz


Saved 3,016 rows -> ../data/processed/facebook_datacenter_longitudinal_2024_2026_policy.csv.gz


Saved 19,852 rows -> ../data/processed/facebook_datacenter_longitudinal_2024_2026_text_deduplicated.csv.gz


Saved 10,597 rows -> ../data/processed/facebook_datacenter_discourse_2026.csv.gz


Saved 1,995 rows -> ../data/processed/facebook_datacenter_discourse_2026_policy.csv.gz


Saved 7,695 rows -> ../data/processed/facebook_datacenter_discourse_2026_text_deduplicated.csv.gz


Saved 1,509 rows -> ../data/processed/facebook_datacenter_discourse_2026_policy_text_deduplicated.csv.gz
Saved 100 rows -> ../output/tables/facebook_manual_validation_longitudinal.csv
Saved 100 rows -> ../output/tables/facebook_manual_validation_2026_discourse.csv


{'raw_rows': 66055,
 'english_rows': 24947,
 'longitudinal': 24915,
 'longitudinal_policy': 3016,
 'discourse_2026': 10597,
 'discourse_2026_policy': 1995,
 'discourse_2026_text_unique': 7695}

## Interpretation, inferential scope, and limitations

This is a keyword-query corpus, not a random sample or a measure of all Facebook discussion. It misses posts using only the spaced spelling. Duplicated and syndicated text can inflate apparent attention, so the pipeline saves broad and normalized-text-deduplicated versions. The 2026 discourse window is a subset of the same consistent query. The review sheets require human completion.